In [ ]:
from __future__ import annotations

import json
import random
import re
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path
from urllib.parse import urljoin, urlparse

import pandas as pd
import requests
from tqdm import tqdm

# request로 부터 crawl하는 코드

In [ ]:
keyword = "폴로"

In [ ]:
# =========================
# Config
# =========================
BASE = "https://www.daangn.com"

UA = (
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
    "AppleWebKit/537.36 (KHTML, like Gecko) "
    "Chrome/122.0.0.0 Safari/537.36"
)

HEADERS_BASE: dict[str, str] = {
    "User-Agent": UA,
    "Accept-Language": "ko-KR,ko;q=0.9,en;q=0.8",
    "Referer": BASE,
}

HEADERS_HTML = {
    **HEADERS_BASE,
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
}

HEADERS_IMG = {
    **HEADERS_BASE,
    "Accept": "image/avif,image/webp,image/apng,image/*,*/*;q=0.8",
}

REMIX_RE = re.compile(
    r"window\.__remixContext\s*=\s*(\{.*?\})\s*;\s*</script>", re.S
)

DEFAULT_TIMEOUT = 20


# =========================
# remix 파싱 로직
# =========================
def extract_remix_context(html: str) -> dict:
    m = REMIX_RE.search(html)
    if not m:
        raise ValueError("remixContext not found")
    return json.loads(m.group(1))


def dig(obj: object, path: list[str]) -> object | None:
    cur = obj
    for p in path:
        if not isinstance(cur, dict) or p not in cur:
            return None
        cur = cur[p]
    return cur


def parse_detail(html: str) -> dict[str, object | None]:
    remix = extract_remix_context(html)

    loader = dig(remix, ["state", "loaderData"])
    if not isinstance(loader, dict):
        raise ValueError("loaderData missing")

    route_key = next((k for k in loader if "buy_sell_id" in k), None)
    if route_key is None:
        raise ValueError("route key missing")

    product = dig(loader, [route_key, "product"])
    if not isinstance(product, dict):
        raise ValueError("product missing")

    user = product.get("user", {}) if isinstance(product.get("user", {}), dict) else {}

    images = product.get("images")
    image_count = len(images) if isinstance(images, list) else None

    image_urls: list[str] = []
    if isinstance(images, list):
        for img in images:
            if isinstance(img, str) and img:
                image_urls.append(img)
            elif isinstance(img, dict):
                u = img.get("url") or img.get("imageUrl") or img.get("src")
                if isinstance(u, str) and u:
                    image_urls.append(u)

    return {
        "favoriteCount": product.get("favoriteCount"),
        "chatCount": product.get("chatCount"),
        "viewCount": product.get("viewCount"),
        "sellerTemperature": user.get("score"),
        "imageCount": image_count,
        "imageUrls": image_urls,
    }


# =========================
# HTTP helpers (보강)
# =========================
def fetch_detail(
    session: requests.Session,
    href: str,
    cookie: str | None = None,
    timeout: int = DEFAULT_TIMEOUT,
) -> dict[str, object | None]:
    url = href if href.startswith("http") else urljoin(BASE, href)

    headers = HEADERS_HTML.copy()
    if cookie:
        headers["Cookie"] = cookie

    r = session.get(url, headers=headers, timeout=timeout)
    r.raise_for_status()
    return parse_detail(r.text)


def infer_ext(full_url: str) -> str:
    ext = Path(urlparse(full_url).path).suffix.lower()
    if ext not in [".jpg", ".jpeg", ".png", ".webp"]:
        ext = ".jpg"
    return ext


def download_main_image(
    session: requests.Session,
    image_url: str,
    item_id: str,
    image_dir: str,
    cookie: str | None = None,
    timeout: int = DEFAULT_TIMEOUT,
) -> tuple[str | None, str | None]:
    if not image_url or not item_id:
        return None, None

    full_url = image_url if image_url.startswith("http") else urljoin(BASE, image_url)
    ext = infer_ext(full_url)

    image_path = Path(image_dir)
    image_path.mkdir(parents=True, exist_ok=True)

    filename = f"{item_id}{ext}"
    save_path = image_path / filename

    # 이미 있으면 스킵
    if save_path.exists() and save_path.stat().st_size > 0:
        return filename, str(save_path)

    headers = HEADERS_IMG.copy()
    if cookie:
        headers["Cookie"] = cookie

    r = session.get(full_url, headers=headers, timeout=timeout, stream=True)
    r.raise_for_status()

    with open(save_path, "wb") as f:
        for chunk in r.iter_content(chunk_size=1024 * 64):
            if chunk:
                f.write(chunk)

    if save_path.stat().st_size == 0:
        save_path.unlink(missing_ok=True)
        return None, None

    return filename, str(save_path)


# =========================
# 병렬 worker (핵심 보강)
# =========================
def _process_one(
    item_id: str,
    href: str,
    cookie: str | None,
    retries: int,
    sleep_min: float,
    sleep_max: float,
    image_dir: str,
    timeout: int,
) -> dict:
    """
    반환 dict는 반드시 id 포함.
    as_completed라 순서가 바뀌어도 merge 가능하게.
    """
    result = {
        "id": item_id,
        "favoriteCount": None,
        "chatCount": None,
        "viewCount": None,
        "sellerTemperature": None,
        "imageCount": None,
        "imageUrls": None,
        "imageFile": None,
        "imagePath": None,
        "status": None,
        "error": None,
    }

    with requests.Session() as session:
        for attempt in range(retries):
            try:
                # ✅ 요청 패턴 완화
                time.sleep(random.uniform(sleep_min, sleep_max))

                data = fetch_detail(session, href, cookie=cookie, timeout=timeout)
                result.update(data)

                image_urls = data.get("imageUrls")
                if isinstance(image_urls, list) and image_urls:
                    image_file, image_path = download_main_image(
                        session=session,
                        image_url=str(image_urls[0]),
                        item_id=item_id,
                        image_dir=image_dir,
                        cookie=cookie,
                        timeout=timeout,
                    )
                    result["imageFile"] = image_file
                    result["imagePath"] = image_path

                result["status"] = "ok"
                return result

            except Exception as e:
                # ✅ backoff + jitter
                backoff = (1.2 ** (attempt + 1)) + random.uniform(0.0, 0.8)
                time.sleep(backoff)
                result["error"] = repr(e)

        result["status"] = "failed"
        return result


def enrich_df_with_detail_parallel(
    df: pd.DataFrame,
    href_col: str = "href",
    id_col: str = "id",
    cookie: str | None = None,
    retries: int = 4,
    timeout: int = DEFAULT_TIMEOUT,
    # ✅ 병렬
    max_workers: int = 8,
    # ✅ 패턴 완화
    sleep_min: float = 0.10,
    sleep_max: float = 0.40,
    # 이미지 저장
    image_dir: str = "../data/images/폴로/",
) -> pd.DataFrame:
    """
    기존 enrich_df_with_detail()의 병렬 강화판.
    - df에 있는 id/href로 상세 파싱 + 메인 이미지 다운로드
    - 결과는 id 기준으로 merge (순서 안정)
    """

    work_df = df[[id_col, href_col]].dropna().copy()
    ids = work_df[id_col].astype(str).tolist()
    hrefs = work_df[href_col].astype(str).tolist()

    futures = []
    rows: list[dict] = []

    with ThreadPoolExecutor(max_workers=max_workers) as ex:
        for item_id, href in zip(ids, hrefs):
            futures.append(
                ex.submit(
                    _process_one,
                    item_id,
                    href,
                    cookie,
                    retries,
                    sleep_min,
                    sleep_max,
                    image_dir,
                    timeout,
                )
            )

        for fut in tqdm(as_completed(futures), total=len(futures), desc="Crawling detail+image"):
            rows.append(fut.result())

    detail_df = pd.DataFrame(rows)

    # ✅ id로 안전 merge (as_completed 때문에 순서가 섞여도 OK)
    merged = df.merge(detail_df, left_on=id_col, right_on="id", how="left", suffixes=("", "_detail"))

    # merge로 생긴 중복 id 컬럼 정리
    if "id_detail" in merged.columns:
        merged.drop(columns=["id_detail"], inplace=True, errors="ignore")

    return merged

In [ ]:
# 쿠키가 있어야 서버로부터 block당하지 않을 확률이 증가함
cookie = r'_gcl_au=1.1.1444086516.1772091281; _ga=GA1.1.1864578218.1772091281; _fbp=fb.1.1772091280970.177480178187518519; _clck=1v9lldb%5E2%5Eg43%5E0%5E2248; search_region=eyJyZWdpb25JZCI6IjYwMzQiLCJyZWdpb25OYW1lIjoi7IK87ISx64%2BZIiwiY291bnRyeUNvZGUiOiJrciJ9; ph_phc_FqtkYFQ4JIDKz1sAraoJA02LALWTkJh8F8okhjnxd3Z_posthog=%7B%22distinct_id%22%3A%22019c98df-0c6b-78ed-9837-b90a124b3fa1%22%2C%22%24sesid%22%3A%5B1772677710535%2C%22019cbbd3-0526-7f55-b1b1-27519dc26a02%22%2C1772677694758%5D%2C%22%24initial_person_info%22%3A%7B%22r%22%3A%22https%3A%2F%2Fwww.google.com%2F%22%2C%22u%22%3A%22https%3A%2F%2Fwww.daangn.com%2Fkr%2F%22%7D%7D; _clsk=h9gh3v%5E1772677711055%5E2%5E0%5Es.clarity.ms%2Fcollect; _ga_KNCHQ0QW6Q=GS2.1.s1772677694$o12$g1$t1772677711$j43$l0$h0'

In [4]:
def bundle_json_to_df(path: str) -> pd.DataFrame:
    with open(path, encoding='utf-8') as f:
        bundle = json.load(f)

    rows = []
    for a in bundle.get('articles', []):
        user = a.get('user') or {}
        region = a.get('region') or {}
        meta = a.get('_meta') or {}

        rows.append(
            {
                'id': a.get('id').split('-')[-1][:-1],
                'href': a.get('href'),
                'price': a.get('price'),
                'title': a.get('title'),
                'status': a.get('status'),
                'content': a.get('content'),
                'createdAt': a.get('createdAt'),
                'boostedAt': a.get('boostedAt'),
                'user_dbId': user.get('dbId'),
                'user_nickname': user.get('nickname'),
                'region_name_from_article': region.get('name'),
                # ✅ 관리용(요청 지역 정보)
                'region_id': meta.get('region_id'),
                'region_name': meta.get('region_name'),
                'region_in': meta.get('region_in'),
                'crawledAt': meta.get('crawledAt'),
            }
        )

    df = pd.DataFrame(rows)
    df['price'] = pd.to_numeric(df['price'], errors='coerce')
    df['createdAt'] = pd.to_datetime(df['createdAt'], errors='coerce')
    df['boostedAt'] = pd.to_datetime(df['boostedAt'], errors='coerce')
    df['crawledAt'] = pd.to_datetime(df['crawledAt'], errors='coerce').dt.tz_localize(
        'Asia/Seoul'
    )
    for col in ['createdAt', 'boostedAt', 'crawledAt']:
        df[col] = pd.to_datetime(df[col], errors='coerce')
        df[col] = df[col].dt.tz_convert('Asia/Seoul')
    return df

In [ ]:
# json파일의 데이터를 df로 불러오기
df = bundle_json_to_df(f'../data/json/search_{keyword}.json')
df.head(1)

,id,href,price,title,status,content,createdAt,boostedAt,user_dbId,user_nickname,region_name_from_article,region_id,region_name,region_in,crawledAt
0,8wqyok7b9sf6,https://www.daangn.com/kr/buy-sell/%ED%8F%B4%E...,71000.0,폴로 랄프로렌 에스테트립 하프 집업팝니다.,Ongoing,구입 후 보관만 한 새상품급 에스테이트립 집업입니다.\n백화점 구입 정품입니다.\n...,2026-02-25 20:51:02.197000+09:00,2026-03-03 17:12:18.814000+09:00,11016644,당근당근당근,삼성동,6034,삼성동,삼성동-6034,2026-03-03 17:19:55+09:00


In [ ]:
merged = enrich_df_with_detail_parallel(
    df,
    href_col="href",
    id_col="id",
    cookie=cookie,
    max_workers=8,        # 6~12 권장
    sleep_min=0.10,
    sleep_max=0.40,
    retries=4,
    image_dir="../data/images/폴로/",
)

merged.to_csv("../data/csv/daangn_폴로.csv", index=False)
merged.to_parquet(f"../data/parquet/daangn_{keyword}.parquet", engine="fastparquet", index=False)

Crawling detail pages: 100%|██████████| 8970/8970 [7:31:43<00:00,  3.02s/it]   


ImportError: Unable to find a usable engine; tried using: 'pyarrow', 'fastparquet'.
A suitable version of pyarrow or fastparquet is required for parquet support.
Trying to import the above resulted in these errors:
 - `Import pyarrow` failed. pyarrow is required for parquet support. Use pip or conda to install the pyarrow package.
 - `Import fastparquet` failed. fastparquet is required for parquet support. Use pip or conda to install the fastparquet package.

In [ ]:
merged.info()

<class 'pandas.DataFrame'>
RangeIndex: 8970 entries, 0 to 8969
Data columns (total 20 columns):
 #   Column                    Non-Null Count  Dtype                     
---  ------                    --------------  -----                     
 0   id                        8970 non-null   str                       
 1   href                      8970 non-null   str                       
 2   price                     8963 non-null   float64                   
 3   title                     8970 non-null   str                       
 4   status                    8970 non-null   str                       
 5   content                   8970 non-null   str                       
 6   createdAt                 8970 non-null   datetime64[us, Asia/Seoul]
 7   boostedAt                 8970 non-null   datetime64[us, Asia/Seoul]
 8   user_dbId                 8970 non-null   str                       
 9   user_nickname             8970 non-null   str                       
 10  region_name

In [ ]:
# parquet파일 읽어오는 방법
df2 = pd.read_parquet(f"../data/parquet/daangn_{keyword}.parquet")

In [ ]:
df2.head(1)

,id,href,price,title,status,content,createdAt,boostedAt,user_dbId,user_nickname,region_name_from_article,region_id,region_name,region_in,crawledAt,favoriteCount,chatCount,viewCount,sellerTemperature,imageCount
0,8wqyok7b9sf6,https://www.daangn.com/kr/buy-sell/%ED%8F%B4%E...,71000.0,폴로 랄프로렌 에스테트립 하프 집업팝니다.,Ongoing,구입 후 보관만 한 새상품급 에스테이트립 집업입니다.\n백화점 구입 정품입니다.\n...,2026-02-25 20:51:02.197000+09:00,2026-03-03 17:12:18.814000+09:00,11016644,당근당근당근,삼성동,6034,삼성동,삼성동-6034,2026-03-03 17:19:55+09:00,11.0,0.0,123.0,99.0,9.0
